In [2]:
!pip install -q torchxrayvision transformers rouge-score evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00eta 0:00:01


In [7]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

DATA_DIR = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university"

# Load metadata
projections = pd.read_csv(f"{DATA_DIR}/indiana_projections.csv")
reports = pd.read_csv(f"{DATA_DIR}/indiana_reports.csv")

# Filter for Frontal views (AP / PA)
frontal = projections[projections['projection'] == 'Frontal']

# Merge image info with reports on study UID
df = pd.merge(frontal, reports, on='uid')

# Clean and combine Findings + Impression
def clean_text(row):
    findings = str(row['findings']) if pd.notna(row['findings']) else ""
    impression = str(row['impression']) if pd.notna(row['impression']) else ""
    text = f"{findings} {impression}".strip()
    return text if len(text) > 10 else None

df['report_text'] = df.apply(clean_text, axis=1)
df = df.dropna(subset=['report_text'])

# Locate image folder (check images/images_normalized or images)
img_dir = os.path.join(DATA_DIR, "images", "images_normalized")
if not os.path.exists(img_dir):
    img_dir = os.path.join(DATA_DIR, "images")

df['image_path'] = df['filename'].apply(lambda x: os.path.join(img_dir, x))

# Filter to make sure the image files actually exist
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

# Split 80% Train, 20% Val
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"✅ Data Preparation Complete!")
print(f"   Total valid samples: {len(df)}")
print(f"   Train samples:       {len(train_df)}")
print(f"   Val samples:         {len(val_df)}")
print("\n--- Sample Radiology Report ---")
print(train_df.iloc[0]['report_text'])


✅ Data Preparation Complete!
   Total valid samples: 3794
   Train samples:       3035
   Val samples:         759

--- Sample Radiology Report ---
No focal consolidation, no definite pleural effusion seen. Exaggerated kyphosis with increased AP dimension of the thorax, curvilinear density projected over the right anterior 3rd and 4th ribs beyond which lung markings are seen XXXX skin fold artifact. Mild aortic ectasia/tortuosity, no typical mediastinal widening to suggest vascular injury. Contour irregularity of the lateral right 9th rib of indeterminate age. 1. No acute cardiopulmonary findings. 2. Age-indeterminate fracture of the right lateral 9th rib. If findings localize to this region, suspect acute fracture.


Cell 3: Dataset & DataLoader

In [9]:
!pip install -q sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 12.1 MB/s eta 0:00:0000:0100:01


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoTokenizer

# Load BioGPT Tokenizer (matches your original project)
print("Loading BioGPT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform or transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert("L")  # Grayscale for TorchXRayVision
        img = self.transform(img)
        return {
            "image": img,
            "report": row['report_text']
        }

def make_collate_fn(tokenizer, max_len=128):
    def collate_fn(batch):
        images = torch.stack([x['image'] for x in batch])
        reports = [x['report'] for x in batch]

        # Tokenize reports
        enc = tokenizer(
            reports,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
            return_attention_mask=True
        )

        return {
            "images": images,
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"]
        }
    return collate_fn

# DataLoaders (batch_size=8 fits well on Kaggle GPU)
collate_fn = make_collate_fn(tokenizer, max_len=128)
train_loader = DataLoader(ChestXRayDataset(train_df), batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(ChestXRayDataset(val_df), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"✅ DataLoaders initialized: {len(train_loader)} training batches.")


Loading BioGPT Tokenizer...
✅ DataLoaders initialized: 380 training batches.


Cell 4: Model Architecture

In [11]:
import torch.nn as nn
import torchxrayvision as xrv
from transformers import AutoModelForCausalLM

class ImageEncoder(nn.Module):
    def __init__(self, xrv_model, resnet_dim=2048, biogpt_dim=1024):
        super().__init__()
        # Feature extractor up to final pooling (outputs [B, 2048, 16, 16])
        self.feature_extractor = nn.Sequential(
            *list(xrv_model.model.children())[:-2]
        )
        
        # Linear projection 2048 → 1024
        self.projection = nn.Sequential(
            nn.Linear(resnet_dim, biogpt_dim),
            nn.LayerNorm(biogpt_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )
    
    def forward(self, images):
        features = self.feature_extractor(images)             # [B, 2048, 16, 16]
        tokens = features.flatten(2).transpose(1, 2)          # [B, 256, 2048]
        tokens = self.projection(tokens)                      # [B, 256, 1024]
        return tokens

class MedicalReportGenerator(nn.Module):
    def __init__(self, num_visual_tokens=256, resnet_dim=2048, biogpt_dim=1024):
        super().__init__()
        self.num_visual_tokens = num_visual_tokens
        
        print("Loading Medical ResNet50 (TorchXRayVision)...")
        xrv_resnet = xrv.models.ResNet(weights="resnet50-res512-all")
        self.image_encoder = ImageEncoder(xrv_resnet, resnet_dim=resnet_dim, biogpt_dim=biogpt_dim)
        
        print("Loading BioGPT Language Model...")
        self.text_decoder = AutoModelForCausalLM.from_pretrained("microsoft/biogpt")
        
        print("\n" + "="*50)
        print("MODEL READY:")
        print(f"  Vision Backbone: ResNet50 (512x512 medical weights)")
        print(f"  Visual Tokens:   {num_visual_tokens} x {biogpt_dim}")
        print(f"  Language Model:  BioGPT")
        print("="*50 + "\n")
    
    def forward(self, images, input_ids, attention_mask):
        B = images.shape[0]
        V = self.num_visual_tokens

        # Pass 1: Encode image and cache K/V pairs
        image_embeds = self.image_encoder(images)
        prefix_mask = torch.ones(B, V, dtype=torch.long, device=images.device)

        with torch.no_grad():
            out_prefix = self.text_decoder(
                inputs_embeds=image_embeds,
                attention_mask=prefix_mask,
                use_cache=True,
                output_hidden_states=False
            )
        past_kv = out_prefix.past_key_values

        # Pass 2: Autoregressively compute text loss conditioned on cached image tokens
        combined_mask = torch.cat([prefix_mask, attention_mask], dim=1)
        labels = input_ids.clone()

        outputs = self.text_decoder(
            input_ids=input_ids,
            attention_mask=combined_mask,
            past_key_values=past_kv,
            labels=labels,
            use_cache=True
        )
        return outputs.loss

    @torch.no_grad()
    def generate(self, images, max_length=128, num_beams=3):
        self.eval()
        image_embeds = self.image_encoder(images)
        return self.text_decoder.generate(
            inputs_embeds=image_embeds,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=self.text_decoder.config.pad_token_id,
            eos_token_id=self.text_decoder.config.eos_token_id
        )

# Initialize Model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MedicalReportGenerator().to(device)

Loading Medical ResNet50 (TorchXRayVision)...
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pc-nih-rsna-siim-vin-resnet50-test512-e400-state.pt -O /root/.torchxrayvision/models_data/pc-nih-rsna-siim-vin-resnet50-test512-e400-state.pt`
[██████████████████████████████████████████████████]
Loading BioGPT Language Model...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.56GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


MODEL READY:
  Vision Backbone: ResNet50 (512x512 medical weights)
  Visual Tokens:   256 x 1024
  Language Model:  BioGPT



Cell 5: Train the Model

In [12]:
import os
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

# Hyperparameters (matches your original config)
num_epochs = 5
learning_rate = 2e-5
gradient_accumulation_steps = 2
save_dir = "/kaggle/working"
best_val_loss = float('inf')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scaler = GradScaler()

print(f"🚀 Starting training for {num_epochs} epochs on device: {device.upper()}\n")

for epoch in range(num_epochs):
    # ==========================
    # TRAIN
    # ==========================
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for step, batch in enumerate(pbar):
        images = batch["images"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        with autocast():
            loss = model(images, input_ids, attention_mask)
            loss = loss / gradient_accumulation_steps
            
        scaler.scale(loss).backward()
        
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        train_loss += loss.item() * gradient_accumulation_steps
        pbar.set_postfix(train_loss=f"{loss.item() * gradient_accumulation_steps:.4f}")
        
    avg_train_loss = train_loss / len(train_loader)

    # ==========================
    # VALIDATION
    # ==========================
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for batch in val_pbar:
            images = batch["images"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            with autocast():
                loss = model(images, input_ids, attention_mask)
            val_loss += loss.item()
            val_pbar.set_postfix(val_loss=f"{loss.item():.4f}")
            
    avg_val_loss = val_loss / len(val_loader)

    print(f"\n📊 Epoch {epoch+1}/{num_epochs} Summary:")
    print(f"   Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Save best checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        checkpoint_path = os.path.join(save_dir, "best_model.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
        }, checkpoint_path)
        print(f"   💾 Saved new best model to {checkpoint_path}\n")
    else:
        print()

🚀 Starting training for 5 epochs on device: CUDA



Epoch 1/5 [Val]: 100%|██████████| 95/95 [01:20<00:00,  1.19it/s, val_loss=0.7639]



📊 Epoch 1/5 Summary:
   Train Loss: 2.2550 | Val Loss: 0.8524
   💾 Saved new best model to /kaggle/working/best_model.pt



Epoch 2/5 [Val]: 100%|██████████| 95/95 [00:26<00:00,  3.55it/s, val_loss=0.6521]



📊 Epoch 2/5 Summary:
   Train Loss: 0.8079 | Val Loss: 0.7040
   💾 Saved new best model to /kaggle/working/best_model.pt



Epoch 3/5 [Val]: 100%|██████████| 95/95 [00:26<00:00,  3.64it/s, val_loss=0.6061]



📊 Epoch 3/5 Summary:
   Train Loss: 0.6906 | Val Loss: 0.6443
   💾 Saved new best model to /kaggle/working/best_model.pt



Epoch 4/5 [Val]: 100%|██████████| 95/95 [00:26<00:00,  3.63it/s, val_loss=0.5973]



📊 Epoch 4/5 Summary:
   Train Loss: 0.6291 | Val Loss: 0.6214
   💾 Saved new best model to /kaggle/working/best_model.pt



Epoch 5/5 [Val]: 100%|██████████| 95/95 [00:26<00:00,  3.62it/s, val_loss=0.5909]



📊 Epoch 5/5 Summary:
   Train Loss: 0.5808 | Val Loss: 0.5976
   💾 Saved new best model to /kaggle/working/best_model.pt



Cell 6: Generate Reports on Test X-Rays (Inference)

In [14]:
import torch

# Load the best checkpoint
checkpoint = torch.load("/kaggle/working/best_model.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Custom generation function using max_new_tokens
def generate_report(model, tokenizer, image_tensor, max_new_tokens=80, num_beams=3):
    model.eval()
    with torch.no_grad():
        image_embeds = model.image_encoder(image_tensor) # [1, 256, 1024]
        prefix_mask = torch.ones(image_embeds.shape[0], 256, dtype=torch.long, device=image_tensor.device)
        
        generated_ids = model.text_decoder.generate(
            inputs_embeds=image_embeds,
            attention_mask=prefix_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=2,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

test_dataset = ChestXRayDataset(val_df)
indices = [5, 12, 25]  # Test sample indices

print("="*80)
print("🩺 RADIOLOGY REPORT GENERATION EVALUATION")
print("="*80)

for idx in indices:
    sample = test_dataset[idx]
    image_tensor = sample["image"].unsqueeze(0).to(device)
    ground_truth = sample["report"]

    generated_text = generate_report(model, tokenizer, image_tensor, max_new_tokens=80, num_beams=3)

    print(f"\n[Case #{idx}]")
    print(f"📋 GROUND TRUTH REPORT:\n{ground_truth}\n")
    print(f"🤖 AI GENERATED REPORT:\n{generated_text}")
    print("-" * 80)

🩺 RADIOLOGY REPORT GENERATION EVALUATION

[Case #5]
📋 GROUND TRUTH REPORT:
Stable cardiomediastinal silhouette. Calcified granuloma in the left lower lobe. Minimal bibasilar airspace disease. No pneumothorax. Degenerative changes of the thoracic spine. Minimal bibasilar airspace disease.

🤖 AI GENERATED REPORT:
able calcified granuloma in the right midlung. No focal consolidation, pneumothorax, or definite pleural effusion identified. Heart size and mediastinal contour within normal limits. Calcified right hilar lymph XXXX. 1. Clear lungs. 2. Stable calcified right mid lung granuloma. 죽탈ᾱ Ζ⍦ ㎟국. Unremarkable XXXXNodular opacity in XXXII.
--------------------------------------------------------------------------------

[Case #12]
📋 GROUND TRUTH REPORT:
Moderate-to-marked enlargement of the cardiac silhouette, mediastinal contours appear similar to prior. Mild bilateral posterior sulcus blunting, interstitial and alveolar opacities greatest in the central lungs and bases with indistinct 

Step 7: Calculate Official Evaluation Metrics (BLEU & ROUGE)

In [15]:
import evaluate
from tqdm import tqdm

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

predictions = []
references = []

eval_samples = 50  # Evaluate first 50 validation cases
print(f"Calculating BLEU and ROUGE on {eval_samples} validation cases...")

for i in tqdm(range(eval_samples)):
    sample = test_dataset[i]
    image_tensor = sample["image"].unsqueeze(0).to(device)
    ground_truth = sample["report"]

    gen_text = generate_report(model, tokenizer, image_tensor, max_new_tokens=60, num_beams=2)
    
    # Simple clean up of any non-ascii artifacts
    clean_gen = "".join([c for c in gen_text if c.isascii()]).strip()
    
    predictions.append(clean_gen)
    references.append([ground_truth])

# Compute scores
bleu_results = bleu.compute(predictions=predictions, references=references)
rouge_results = rouge.compute(predictions=predictions, references=references)

print("\n" + "="*50)
print("🏆 OFFICIAL EVALUATION METRICS:")
print("="*50)
print(f"  BLEU-1:  {bleu_results['precisions'][0]:.4f}")
print(f"  BLEU-2:  {bleu_results['precisions'][1]:.4f}")
print(f"  BLEU-3:  {bleu_results['precisions'][2]:.4f}")
print(f"  BLEU-4:  {bleu_results['precisions'][3]:.4f}")
print(f"  ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_results['rougeL']:.4f}")
print("="*50)

Calculating BLEU and ROUGE on 50 validation cases...


100%|██████████| 50/50 [01:02<00:00,  1.24s/it]



🏆 OFFICIAL EVALUATION METRICS:
  BLEU-1:  0.3679
  BLEU-2:  0.1293
  BLEU-3:  0.0513
  BLEU-4:  0.0212
  ROUGE-1: 0.3392
  ROUGE-2: 0.1025
  ROUGE-L: 0.2251
